In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from pandas_datareader import data as pdr

In [2]:
## Pulling ticker data for later use ## 
portfolio_groups = {
    "Dom_Eq": ['SPY', 'QQQ', 'IWM'],
    "Intl_Eq": ['EFA', 'EEM'],
    "Fixed_Inc": ['AGG', 'TLT', 'LQD'],
    "Alt": ['GLD', 'VNQ'],
    "Factor": ['MTUM', 'VLUE', 'QUAL', 'USMV']
}
all_tickers = [ticker for group in portfolio_groups.values() for ticker in group]
portfolio_data : pd.DataFrame = yf.download(all_tickers, # type: ignore
                                            period = "5y",
                                            auto_adjust= False,
                                            group_by="column")
if portfolio_data is None:
    raise ValueError("Download failed")
adj_close = portfolio_data["Adj Close"] 
daily_returns = adj_close.pct_change().dropna()


[*********************100%***********************]  14 of 14 completed


In [3]:
start = daily_returns.index.min()
end = daily_returns.index.max()
rfr = pdr.DataReader("DGS3MO", "fred", start=start, end = end)
rfr['risk_free_rate'] = rfr['DGS3MO'] / 100
rfr['daily_rfr'] = rfr['risk_free_rate'] / 252
daily_rfr = rfr['daily_rfr'].reindex(daily_returns.index).ffill()


In [5]:
## Pulling Metadata for later use ##
ticker_metadata = {
    ticker:group
    for group, tickers in portfolio_groups.items()
    for ticker in tickers
}
ticker_meta_df = (
    pd.DataFrame.from_dict(ticker_metadata, orient = "index", columns = ['group'])
    .reset_index()
    .rename(columns = {"index":"ticker"})
)

In [6]:
beta_series = pd.Series(index = daily_returns.columns, dtype = float)

benchmark_var = daily_returns['SPY'].var()

for col in daily_returns.columns:
    cov = daily_returns[col].cov(daily_returns['SPY'])
    beta = cov / benchmark_var
    beta_series[col] = beta

beta_series

Ticker
AGG     0.074552
EEM     0.770666
EFA     0.773727
GLD     0.154507
IWM     1.115075
LQD     0.186999
MTUM    1.074152
QQQ     1.261736
QUAL    0.989070
SPY     1.000000
TLT     0.070126
USMV    0.595172
VLUE    0.907880
VNQ     0.731063
dtype: float64

In [7]:
## Feature Engineering ## 
daily_mean_returns = daily_returns.mean()
annual_expected_returns = daily_returns.mean() * 252
average_trade_volume = portfolio_data['Volume'].mean()
annual_volatility = daily_returns.std() * np.sqrt(252)
corr_matrix = daily_returns.corr()
cov_matrix = daily_returns.cov() * 252
rolling_vol = daily_returns.rolling(30).std() * np.sqrt(252)
cumulative_returns = (1 + daily_returns).cumprod()
running_max = cumulative_returns.cummax()
drawdowns = (cumulative_returns - running_max) / running_max
portfolio_metrics = {
    "returns": annual_expected_returns,
    "volatility": annual_volatility,
    "beta": beta_series,
    "correlation": corr_matrix,
    "covariance": cov_matrix,
    "drawdowns": drawdowns
}

In [8]:
metrics_table = pd.DataFrame()
metrics_table['expected_returns'] = daily_mean_returns.round(6)
metrics_table['ann_returns'] = annual_expected_returns
metrics_table['annual_volatility'] = annual_volatility
metrics_table['beta'] = beta_series
metrics_table['av_trade_volume'] = average_trade_volume
metrics_table['ann_sharpe'] = (
    daily_returns.sub(daily_rfr, axis=0).mean()*252) / (daily_returns.std()*(252**0.5))
metrics_table['max_drawdowns'] = drawdowns.min()


In [9]:
portfolio_data.to_csv("data/processed/portfolio_data.csv")
daily_returns.to_csv("data/processed/daily_returns.csv")
adj_close.to_csv("data/processed/adj_close.csv")
cov_matrix.to_csv("data/processed/cov_matrix.csv")
metrics_table.to_csv("data/processed/metrics_table.csv")
daily_rfr.to_csv("data/processed/daily_risk_free_rate.csv")